# News MCP Query Playground

Use `query_news` against the local vector store.

Before running these cells, make sure you have synced feeds, embedded news, and populated Qdrant.

In [1]:
from pathlib import Path
import sys

from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
load_dotenv(repo_root / ".env", override=True)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

PosixPath('/Users/jackson/Codes/self/JarvisTradingLabs/open-news-mcp')

In [2]:
import asyncio
import json

from src.config import settings
from src.tools.query import query_news

In [3]:
{
    "embedding_backend": settings.embedding_backend,
    "embedding_model": settings.embedding_model,
    "embedding_device": settings.embedding_device,
    "vector_backend": settings.vector_backend,
    "vector_collection": settings.vector_collection,
}

{'embedding_backend': 'local',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_device': 'cpu',
 'vector_backend': 'qdrant',
 'vector_collection': 'news_articles'}

In [4]:
async def run_query(**kwargs):
    raw = await query_news(**kwargs)
    data = json.loads(raw)
    if not data.get("ok"):
        return data

    return {
        "ok": True,
        "count": data.get("count"),
        "articles": [
            {
                "id": article.get("id"),
                "name": article.get("name"),
                "title": article.get("title"),
                "score": article.get("score"),
                "tier": article.get("tier"),
                "category": article.get("category"),
                "published_at": article.get("published_at"),
                "url": article.get("url"),
            }
            for article in data.get("articles", [])
        ],
    }


async def demo():
    return {
        "macro": await run_query(query="federal reserve interest rates inflation", timespan="30d", limit=5),
        "energy": await run_query(query="opec oil production cuts crude prices", timespan="30d", limit=5),
    }


await demo()

/Users/jackson/Codes/self/JarvisTradingLabs/open-news-mcp/src/vector/providers/qdrant.py:40: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(
/Users/jackson/Codes/self/JarvisTradingLabs/open-news-mcp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14042.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'macro': {'ok': True,
  'count': 5,
  'articles': [{'id': 579,
    'name': 'Economic Data',
    'title': 'Will US inflation change the outlook for interest rates? - Financial Times',
    'score': 0.6701731,
    'tier': 2,
    'category': 'economic',
    'published_at': '2026-03-08T12:00:06Z',
    'url': 'https://news.google.com/rss/articles/CBMicEFVX3lxTE0wd3RwenNOS2NKSFJHaW9pSmFNdUpYNVhJemdzb1pKaEhCVlVvR0dYNld2WEFhWUVtOWtpYndHaGVtSXFKT1JjZURmNEhUZDQ4M2FsYzhWcWpJclFPaGhXb3BkTjdhMGdZaXltWkJCQ0c?oc=5'},
   {'id': 277,
    'name': 'Central Bank Rates',
    'title': "Amid oil shock uncertainty, Fed's Hammack says central bank must lower inflation - Reuters",
    'score': 0.56565547,
    'tier': 2,
    'category': 'forex',
    'published_at': '2026-03-06T23:30:45Z',
    'url': 'https://news.google.com/rss/articles/CBMiywFBVV95cUxNQXJlNUJxaDNmRWcyZDQyeTQ5aTU5alpvWHNXTlRPc0l1X29XTkQ2YnpQTXQ0VXhtRWVnWlNZRXZDSXRhZG5BNTAyWHRhTk9yYU1YMVM5TGFzTUVLcDFMcXVBLXh1THBEWmNHTl9DOVFjZWZEa3N1ZzFZUGVWSzRqSH

## Macro Query

In [5]:
await run_query(query="federal reserve rate cuts inflation outlook", timespan="30d", limit=10)

{'ok': True,
 'count': 10,
 'articles': [{'id': 579,
   'name': 'Economic Data',
   'title': 'Will US inflation change the outlook for interest rates? - Financial Times',
   'score': 0.6412691,
   'tier': 2,
   'category': 'economic',
   'published_at': '2026-03-08T12:00:06Z',
   'url': 'https://news.google.com/rss/articles/CBMicEFVX3lxTE0wd3RwenNOS2NKSFJHaW9pSmFNdUpYNVhJemdzb1pKaEhCVlVvR0dYNld2WEFhWUVtOWtpYndHaGVtSXFKT1JjZURmNEhUZDQ4M2FsYzhWcWpJclFPaGhXb3BkTjdhMGdZaXltWkJCQ0c?oc=5'},
  {'id': 584,
   'name': 'Economic Data',
   'title': 'Fed rate cut bets pushed back as Iran war raises inflation risks - Anadolu Ajansı',
   'score': 0.5994414,
   'tier': 2,
   'category': 'economic',
   'published_at': '2026-03-08T07:11:40Z',
   'url': 'https://news.google.com/rss/articles/CBMiqAFBVV95cUxOLVkwWjV3MjUyeDZUX21mSGlKcFhGaU1hY1VuZWdzNFM0YVlhMWhObzRRLW1nS0g0Y2JOc0RpVzQ1VVIzcnRBN3JicDFjXzRlekdvaGE4d1JVNVFlOXZkS2pNQ3VwX3hWYjhWbUVMc0pIUkszNzNYaUdrbGJXSHJyRnVucWl6NlZOU3ZVX04wdURYYXYyVjNwdXR1dUVN

## Filtered By Category And Tier

In [6]:
await run_query(
    query="tariffs china trade war commodities",
    categories=["markets"],
    tiers=[1, 2],
    timespan="14d",
    limit=10,
)

{'ok': True,
 'count': 1,
 'articles': [{'id': 85,
   'name': 'Bloomberg Markets',
   'title': 'China Eyes ‘Landmark’ Year for US Ties, Urges End to Iran War - Bloomberg.com',
   'score': 0.45872656,
   'tier': 1,
   'category': 'markets',
   'published_at': '2026-03-08T09:22:41Z',
   'url': 'https://news.google.com/rss/articles/CBMirgFBVV95cUxPcG5pTnktWDI1QzRMVnBWaElOZnNFVGt6RVNCR05IdzRPdUk5Uno4S3NLdlNhMmI1cFczTWpEOC00SkNCc0xyQko3VXQ3akNVSjV2Z2o1Vm5ZSE0yRTBkYXltXzVVal93T1NBYk9nNDF6UUZuSkMxcWVJbGt6elR0RlVxTktKU0pteGRDN0NMVU40Y21hcDN0eS02eFczZk4xS2ZWb1J5a1JqclFuSGc?oc=5'}]}

## Source-Scoped Semantic Query

In [7]:
await run_query(
    query="bank regulation and monetary policy signals",
    sources=["Federal Reserve", "ECB Press"],
    timespan="60d",
    limit=10,
)

{'ok': True,
 'count': 4,
 'articles': [{'id': 428,
   'name': 'Federal Reserve',
   'title': 'Federal Reserve Board finalizes hypothetical scenarios for its annual stress test and votes to maintain the current stress test-related capital requirements until public feedback can be considered',
   'score': 0.48718297,
   'tier': 1,
   'category': 'centralbanks',
   'published_at': '2026-02-04T21:30:00Z',
   'url': 'https://www.federalreserve.gov/newsevents/pressreleases/bcreg20260204a.htm'},
  {'id': 432,
   'name': 'Federal Reserve',
   'title': 'Federal Open Market Committee reaffirms its "Statement on Longer-Run Goals and Monetary Policy Strategy"',
   'score': 0.48183537,
   'tier': 1,
   'category': 'centralbanks',
   'published_at': '2026-01-28T19:00:00Z',
   'url': 'https://www.federalreserve.gov/newsevents/pressreleases/monetary20260128b.htm'},
  {'id': 425,
   'name': 'Federal Reserve',
   'title': 'Minutes of the Federal Open Market Committee, January 27–28, 2026',
   'score': 

## Empty Query Error Example

In [8]:
await run_query(query="", timespan="7d", limit=3)

{'ok': False,
 'error': {'code': 'INVALID_ARGUMENT',
  'message': 'query is required.',
  'field': 'query'}}